In [1]:
# ============================================================
# evaluate_cnnbilstm_v7_a6000_random_subject_split.py
#
# CNN-BiLSTM downstream HAR evaluation for your v7 synthetic data.
#
# Main target:
#   NVIDIA RTX A6000 48GB
#
# Channel switch:
#   CHANNEL_MODE = "6ch" -> ACC_x, ACC_y, ACC_z, BVP, EDA, TEMP
#   CHANNEL_MODE = "4ch" -> ACC_x, ACC_y, ACC_z, BVP
#
# Subject split:
#   Random subject-wise split with fixed SEED.
#   With 15 subjects:
#       Train = 10 subjects
#       Val   = 2 subjects
#       Test  = 3 subjects
#
# Input real:
#   processed_all_subjects_64hz/all_X.npy
#   processed_all_subjects_64hz/all_y.npy
#   processed_all_subjects_64hz/all_subject.npy
#
# Input synthetic:
#   all6_average_synthetic_subject_outputs/generated_subjects_all_X.npy
#   all6_average_synthetic_subject_outputs/generated_subjects_all_y.npy
#   all6_average_synthetic_subject_outputs/generated_subjects_all_subject.npy
#
# Experiments:
#   1. Real train -> Real test
#   2. Real train -> Synthetic test
#   3. Synthetic train -> Real test
#   4. Real + Synthetic train -> Real test
#
# Outputs:
#   cnnbilstm_v7_results_a6000_6ch/
#   or
#   cnnbilstm_v7_results_a6000_4ch/
# ============================================================

import copy
import random
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support,
)


# ============================================================
# Main switch
# ============================================================

# Choose one:
CHANNEL_MODE = "6ch"
# CHANNEL_MODE = "4ch"


# ============================================================
# Paths
# ============================================================

PROJECT_ROOT = Path("/home/iailab42/khans1/projects")

REAL_DIR = PROJECT_ROOT / "data"
SYN_DIR = PROJECT_ROOT / "data" / "synthetic"

REAL_X_PATH = REAL_DIR / "all_X.npy"
REAL_Y_PATH = REAL_DIR / "all_y.npy"
REAL_SUBJECT_PATH = REAL_DIR / "all_subject.npy"

SYN_X_PATH = SYN_DIR / "all_X_synthetic.npy"
SYN_Y_PATH = SYN_DIR / "all_y_synthetic.npy"
SYN_SUBJECT_PATH = SYN_DIR / "all_subject_synthetic.npy"

# SYN_X_PATH = SYN_DIR / "timevae"/ "all_X_synthetic.npy"
# SYN_Y_PATH = SYN_DIR / "timevae"/ "all_y_synthetic.npy"
# SYN_SUBJECT_PATH = SYN_DIR / "timevae"/ "all_subject_synthetic.npy"
# SYN_METADATA_PATH = SYN_DIR / "timevae"/ "all_metadata_synthetic.csv"

OUT_DIR = PROJECT_ROOT / "results" / f"cnnbilstm_kovae_downstream_{CHANNEL_MODE}"
# OUT_DIR = PROJECT_ROOT / "results" / f"cnnbilstm_timevae_downstream_{CHANNEL_MODE}"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# General config
# ============================================================

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ALL_CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]

if CHANNEL_MODE == "6ch":
    CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]
elif CHANNEL_MODE == "4ch":
    CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP"]
else:
    raise ValueError("CHANNEL_MODE must be '6ch' or '4ch'.")

USE_CHANNEL_INDICES = [ALL_CHANNEL_NAMES.index(ch) for ch in CHANNEL_NAMES]
NUM_CHANNELS = len(CHANNEL_NAMES)

ACTIVITY_IDS = [1, 2, 3, 4, 5, 6, 7, 8]
NUM_CLASSES = len(ACTIVITY_IDS)

WINDOW_LEN = 512

# Paper-style subsequence setup adapted to your 512-sample windows:
# paper: 128 = 4 x 32
# yours: 512 = 4 x 128
N_SEQ = 4
N_STEPS = WINDOW_LEN // N_SEQ

if WINDOW_LEN % N_SEQ != 0:
    raise ValueError("WINDOW_LEN must be divisible by N_SEQ.")

# Random subject-wise split counts.
N_TEST_SUBJECTS = 3
N_VAL_SUBJECTS = 2

# Leave empty for random split.
# Or manually set, e.g. TEST_SUBJECTS = ["S13", "S14", "S15"]
TEST_SUBJECTS = []
VAL_SUBJECTS = []


# ============================================================
# A6000 training settings
# ============================================================

BATCH_SIZE = 384
NUM_WORKERS = 6

EPOCHS_REAL_TO_REAL = 100
EPOCHS_REAL_TO_SYN = 100
EPOCHS_SYN_TO_REAL = 100
EPOCHS_REAL_PLUS_SYN_TO_REAL = 100

LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.5

USE_CLASS_WEIGHTS = True
USE_AMP = True
USE_TF32 = True

SAVE_CONFUSION_MATRIX_PNG = True

# Early stopping based on validation Macro-F1.
PATIENCE = 15


# ============================================================
# CUDA backend settings
# ============================================================

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = USE_TF32
    torch.backends.cudnn.allow_tf32 = USE_TF32

    # More reproducible. You can set True for speed, but False is safer.
    torch.backends.cudnn.benchmark = False


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


def autocast_context():
    """
    New non-deprecated AMP autocast syntax.

    Old deprecated:
        torch.cuda.amp.autocast()

    New:
        torch.amp.autocast("cuda", enabled=True/False)
    """
    if DEVICE == "cuda":
        return torch.amp.autocast("cuda", enabled=USE_AMP)

    return nullcontext()


# ============================================================
# Label helpers
# ============================================================

LABEL_TO_INDEX = {label: i for i, label in enumerate(ACTIVITY_IDS)}
INDEX_TO_LABEL = {i: label for label, i in LABEL_TO_INDEX.items()}


def labels_to_indices(y):
    y = np.asarray(y).astype(np.int64)
    out = np.zeros_like(y, dtype=np.int64)

    for i, lab in enumerate(y):
        lab = int(lab)

        if lab not in LABEL_TO_INDEX:
            raise ValueError(f"Unexpected label {lab}. Expected labels: {ACTIVITY_IDS}")

        out[i] = LABEL_TO_INDEX[lab]

    return out


def indices_to_labels(y_idx):
    y_idx = np.asarray(y_idx).astype(np.int64)
    return np.array([INDEX_TO_LABEL[int(i)] for i in y_idx], dtype=np.int64)


# ============================================================
# File helpers
# ============================================================

def require_file(path):
    path = Path(path)

    if not path.exists():
        error_message = (
            f"\nMissing file: {path}\n"
            f"Make sure these files exist:\n"
            f"  /home/iailab42/khans1/projects/data/all_X.npy\n"
            f"  /home/iailab42/khans1/projects/data/all_y.npy\n"
            f"  /home/iailab42/khans1/projects/data/all_subject.npy\n"
            f"  /home/iailab42/khans1/projects/data/synthetic/all_X_synthetic.npy\n"
            f"  /home/iailab42/khans1/projects/data/synthetic/all_y_synthetic.npy\n"
            f"  /home/iailab42/khans1/projects/data/synthetic/all_subject_synthetic.npy\n"
        )
        raise FileNotFoundError(error_message)

    return path


# ============================================================
# Load data
# ============================================================

def load_data(x_path, y_path, subject_path, name):
    require_file(x_path)
    require_file(y_path)
    require_file(subject_path)

    X = np.load(x_path).astype(np.float32)
    y = np.load(y_path).astype(np.int64)
    subjects = np.load(subject_path, allow_pickle=True).astype(str)

    if X.ndim != 3:
        raise ValueError(f"{name} X should be [N, T, C], got {X.shape}")

    if X.shape[1] != WINDOW_LEN:
        raise ValueError(
            f"{name} expected window length {WINDOW_LEN}, got {X.shape[1]}"
        )

    if X.shape[2] != len(ALL_CHANNEL_NAMES):
        raise ValueError(
            f"{name} expected original 6 channels, got {X.shape[2]}"
        )

    if len(X) != len(y):
        raise ValueError(f"{name} X/y mismatch: {len(X)} vs {len(y)}")

    if len(y) != len(subjects):
        raise ValueError(f"{name} y/subjects mismatch: {len(y)} vs {len(subjects)}")

    keep = np.isin(y, np.array(ACTIVITY_IDS, dtype=np.int64))

    X = X[keep]
    y = y[keep]
    subjects = subjects[keep]

    # Channel switch happens here.
    X = X[:, :, USE_CHANNEL_INDICES].astype(np.float32)

    print("\n" + "=" * 70)
    print(f"Loaded {name}")
    print("=" * 70)
    print("X:", X.shape)
    print("y:", y.shape)
    print("subjects:", subjects.shape)
    print("CHANNEL_MODE:", CHANNEL_MODE)
    print("Using channels:", CHANNEL_NAMES)
    print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))
    print("Subject counts:", dict(zip(*np.unique(subjects, return_counts=True))))

    return X, y, subjects


def load_real_data():
    return load_data(
        REAL_X_PATH,
        REAL_Y_PATH,
        REAL_SUBJECT_PATH,
        name="real data",
    )


def load_synthetic_data():
    return load_data(
        SYN_X_PATH,
        SYN_Y_PATH,
        SYN_SUBJECT_PATH,
        name="synthetic data",
    )


# ============================================================
# Subject split
# ============================================================

def subject_sort_key(s):
    s = str(s)

    if s.startswith("S"):
        try:
            return int(s[1:])
        except Exception:
            return s

    return s


def make_subject_split(real_subjects):
    unique_subjects = sorted(np.unique(real_subjects).astype(str), key=subject_sort_key)

    if len(unique_subjects) < (N_TEST_SUBJECTS + N_VAL_SUBJECTS + 1):
        raise ValueError(
            f"Not enough real subjects for split. Found: {unique_subjects}"
        )

    # Manual split if user fills TEST_SUBJECTS / VAL_SUBJECTS.
    if len(TEST_SUBJECTS) > 0 or len(VAL_SUBJECTS) > 0:
        if len(TEST_SUBJECTS) == 0 or len(VAL_SUBJECTS) == 0:
            raise ValueError(
                "If using manual split, set both TEST_SUBJECTS and VAL_SUBJECTS."
            )

        test_subjects = list(TEST_SUBJECTS)
        val_subjects = list(VAL_SUBJECTS)

        train_subjects = [
            s for s in unique_subjects
            if s not in test_subjects and s not in val_subjects
        ]

        split_mode = "manual"

    else:
        # Fixed random subject split.
        rng = np.random.default_rng(SEED)
        shuffled_subjects = np.array(unique_subjects, dtype=object)
        rng.shuffle(shuffled_subjects)
        shuffled_subjects = shuffled_subjects.tolist()

        test_subjects = shuffled_subjects[:N_TEST_SUBJECTS]
        val_subjects = shuffled_subjects[
            N_TEST_SUBJECTS:N_TEST_SUBJECTS + N_VAL_SUBJECTS
        ]
        train_subjects = shuffled_subjects[N_TEST_SUBJECTS + N_VAL_SUBJECTS:]

        split_mode = f"random_fixed_seed_{SEED}"

    train_subjects = sorted(train_subjects, key=subject_sort_key)
    val_subjects = sorted(val_subjects, key=subject_sort_key)
    test_subjects = sorted(test_subjects, key=subject_sort_key)

    if len(set(train_subjects) & set(val_subjects)) > 0:
        raise ValueError("Train/val subject overlap detected.")

    if len(set(train_subjects) & set(test_subjects)) > 0:
        raise ValueError("Train/test subject overlap detected.")

    if len(set(val_subjects) & set(test_subjects)) > 0:
        raise ValueError("Val/test subject overlap detected.")

    print("\n" + "=" * 70)
    print("Real subject split")
    print("=" * 70)
    print("Split mode:     ", split_mode)
    print("Train subjects: ", train_subjects)
    print("Val subjects:   ", val_subjects)
    print("Test subjects:  ", test_subjects)
    print("Train % by subject:", len(train_subjects) / len(unique_subjects) * 100.0)
    print("Val % by subject:  ", len(val_subjects) / len(unique_subjects) * 100.0)
    print("Test % by subject: ", len(test_subjects) / len(unique_subjects) * 100.0)

    split_df = pd.DataFrame({
        "split": ["train", "val", "test"],
        "subjects": [
            ",".join(train_subjects),
            ",".join(val_subjects),
            ",".join(test_subjects),
        ],
        "num_subjects": [
            len(train_subjects),
            len(val_subjects),
            len(test_subjects),
        ],
        "percentage_by_subject": [
            len(train_subjects) / len(unique_subjects) * 100.0,
            len(val_subjects) / len(unique_subjects) * 100.0,
            len(test_subjects) / len(unique_subjects) * 100.0,
        ],
        "split_mode": [split_mode, split_mode, split_mode],
        "seed": [SEED, SEED, SEED],
    })

    split_df.to_csv(OUT_DIR / "real_subject_split.csv", index=False)

    return train_subjects, val_subjects, test_subjects


def filter_by_subjects(X, y, subjects, selected_subjects):
    selected_subjects = set([str(s) for s in selected_subjects])
    mask = np.array([str(s) in selected_subjects for s in subjects], dtype=bool)

    return X[mask], y[mask], subjects[mask]


# ============================================================
# Dataset / DataLoader
# ============================================================

class HARWindowDataset(Dataset):
    def __init__(self, X, y_original):
        X = np.asarray(X, dtype=np.float32)
        y_original = np.asarray(y_original, dtype=np.int64)
        y_idx = labels_to_indices(y_original)

        if X.ndim != 3:
            raise ValueError(f"Expected X [N,T,C], got {X.shape}")

        if X.shape[1] != WINDOW_LEN:
            raise ValueError(f"Expected window length {WINDOW_LEN}, got {X.shape[1]}")

        if X.shape[2] != NUM_CHANNELS:
            raise ValueError(f"Expected {NUM_CHANNELS} channels, got {X.shape[2]}")

        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y_idx).long()
        self.y_original = y_original

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loader(X, y, batch_size, shuffle):
    dataset = HARWindowDataset(X, y)

    # Avoid BatchNorm crash from final train batch of size 1.
    drop_last = bool(shuffle and len(dataset) > batch_size)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=drop_last,
        persistent_workers=(NUM_WORKERS > 0),
    )

    return loader


def compute_class_weights(y_original):
    y_idx = labels_to_indices(y_original)
    counts = np.bincount(y_idx, minlength=NUM_CLASSES).astype(np.float32)

    counts = np.maximum(counts, 1.0)

    weights = counts.sum() / (NUM_CLASSES * counts)
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32)


# ============================================================
# CNN-BiLSTM model
# ============================================================

class TimeDistributedConvBranch(nn.Module):
    def __init__(self, in_channels, kernel_size, dropout):
        super().__init__()

        padding = kernel_size // 2

        self.net = nn.Sequential(
            nn.Conv1d(
                in_channels=in_channels,
                out_channels=64,
                kernel_size=kernel_size,
                padding=padding,
            ),
            nn.ReLU(inplace=True),

            nn.Conv1d(
                in_channels=64,
                out_channels=32,
                kernel_size=kernel_size,
                padding=padding,
            ),
            nn.ReLU(inplace=True),

            nn.Dropout(dropout),
            nn.MaxPool1d(kernel_size=2),
        )

    def forward(self, x):
        # x: [B, S, steps, C]
        B, S, steps, C = x.shape

        # TimeDistributed Conv1D:
        # [B, S, steps, C] -> [B*S, C, steps]
        x = x.reshape(B * S, steps, C)
        x = x.transpose(1, 2)

        h = self.net(x)

        # h: [B*S, 32, steps/2]
        h = h.reshape(B, S, -1)

        return h


class MultiBranchCNNBiLSTM(nn.Module):
    def __init__(
        self,
        num_channels,
        num_classes,
        n_seq,
        n_steps,
        dropout,
    ):
        super().__init__()

        self.num_channels = num_channels
        self.num_classes = num_classes
        self.n_seq = n_seq
        self.n_steps = n_steps

        self.branch_k3 = TimeDistributedConvBranch(
            in_channels=num_channels,
            kernel_size=3,
            dropout=dropout,
        )

        self.branch_k7 = TimeDistributedConvBranch(
            in_channels=num_channels,
            kernel_size=7,
            dropout=dropout,
        )

        self.branch_k11 = TimeDistributedConvBranch(
            in_channels=num_channels,
            kernel_size=11,
            dropout=dropout,
        )

        branch_out_dim = 32 * (n_steps // 2)
        lstm_input_dim = branch_out_dim * 3

        self.bilstm1 = nn.LSTM(
            input_size=lstm_input_dim,
            hidden_size=64,
            batch_first=True,
            bidirectional=True,
        )

        self.bilstm2 = nn.LSTM(
            input_size=64 * 2,
            hidden_size=32,
            batch_first=True,
            bidirectional=True,
        )

        self.fc = nn.Linear(32 * 2, 128)
        self.bn = nn.BatchNorm1d(128)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: [B, 512, C]
        B, T, C = x.shape

        expected_t = self.n_seq * self.n_steps

        if T != expected_t:
            raise ValueError(f"Expected T={expected_t}, got T={T}")

        if C != self.num_channels:
            raise ValueError(f"Expected C={self.num_channels}, got C={C}")

        # [B, 512, C] -> [B, 4, 128, C]
        x = x.reshape(B, self.n_seq, self.n_steps, C)

        h3 = self.branch_k3(x)
        h7 = self.branch_k7(x)
        h11 = self.branch_k11(x)

        h = torch.cat([h3, h7, h11], dim=-1)

        h, _ = self.bilstm1(h)
        h, (h_n, _) = self.bilstm2(h)

        # h_n shape: [num_layers * num_directions, B, hidden]
        # last forward = h_n[-2], last backward = h_n[-1]
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)

        z = self.fc(h_final)
        z = self.bn(z)
        z = F.relu(z)
        z = self.dropout(z)

        logits = self.classifier(z)

        return logits


def make_new_model():
    model = MultiBranchCNNBiLSTM(
        num_channels=NUM_CHANNELS,
        num_classes=NUM_CLASSES,
        n_seq=N_SEQ,
        n_steps=N_STEPS,
        dropout=DROPOUT,
    )

    return model


# ============================================================
# Metrics
# ============================================================

def evaluate_model(model, loader):
    model.eval()

    all_true = []
    all_pred = []
    all_prob = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            with autocast_context():
                logits = model(X_batch)

            probs = torch.softmax(logits.float(), dim=1)
            pred = torch.argmax(probs, dim=1)

            all_true.append(y_batch.cpu().numpy())
            all_pred.append(pred.cpu().numpy())
            all_prob.append(probs.cpu().numpy())

    y_true_idx = np.concatenate(all_true)
    y_pred_idx = np.concatenate(all_pred)
    y_prob = np.concatenate(all_prob)

    y_true = indices_to_labels(y_true_idx)
    y_pred = indices_to_labels(y_pred_idx)

    return y_true, y_pred, y_prob


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    accuracy = accuracy_score(y_true, y_pred)

    macro_precision = precision_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="macro",
        zero_division=0,
    )

    macro_recall = recall_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="macro",
        zero_division=0,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="macro",
        zero_division=0,
    )

    weighted_precision = precision_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="weighted",
        zero_division=0,
    )

    weighted_recall = recall_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="weighted",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="weighted",
        zero_division=0,
    )

    # Balanced accuracy for multiclass = average recall across classes.
    # This is intentionally same definition as macro recall.
    balanced_accuracy = macro_recall

    metrics = {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
        "balanced_accuracy": balanced_accuracy,
    }

    return metrics


def compute_per_class_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred, labels=ACTIVITY_IDS)

    row_sum = cm.sum(axis=1)
    diag = np.diag(cm)

    per_class_accuracy = np.divide(
        diag,
        row_sum,
        out=np.zeros_like(diag, dtype=np.float64),
        where=row_sum != 0,
    )

    rows = []

    for i, act in enumerate(ACTIVITY_IDS):
        rows.append({
            "activity": act,
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "per_class_accuracy_same_as_recall": float(per_class_accuracy[i]),
            "support": int(support[i]),
        })

    per_class_df = pd.DataFrame(rows)

    return per_class_df, cm


# ============================================================
# Plot / save results
# ============================================================

def save_confusion_matrix_png(cm, experiment_name):
    fig, ax = plt.subplots(figsize=(8, 7))

    im = ax.imshow(cm, interpolation="nearest")
    fig.colorbar(im, ax=ax)

    ax.set_title(f"Confusion Matrix | {experiment_name}")
    ax.set_xlabel("Predicted activity")
    ax.set_ylabel("True activity")

    tick_marks = np.arange(len(ACTIVITY_IDS))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels([str(a) for a in ACTIVITY_IDS])
    ax.set_yticklabels([str(a) for a in ACTIVITY_IDS])

    threshold = cm.max() / 2.0 if cm.max() > 0 else 0.5

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = int(cm[i, j])
            ax.text(
                j,
                i,
                str(value),
                ha="center",
                va="center",
                color="white" if value > threshold else "black",
                fontsize=9,
            )

    plt.tight_layout()

    save_path = OUT_DIR / f"{experiment_name}_confusion_matrix.png"
    plt.savefig(save_path, dpi=150)
    plt.close()

    print("Saved:", save_path)


def save_final_results(model, test_loader, experiment_name):
    y_true, y_pred, y_prob = evaluate_model(model, test_loader)

    metrics = compute_metrics(y_true, y_pred)
    per_class_df, cm = compute_per_class_metrics(y_true, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{a}" for a in ACTIVITY_IDS],
        columns=[f"pred_{a}" for a in ACTIVITY_IDS],
    )

    pred_df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "correct": y_true == y_pred,
    })

    for i, act in enumerate(ACTIVITY_IDS):
        pred_df[f"prob_activity_{act}"] = y_prob[:, i]

    per_class_path = OUT_DIR / f"{experiment_name}_per_class_metrics.csv"
    cm_path = OUT_DIR / f"{experiment_name}_confusion_matrix.csv"
    pred_path = OUT_DIR / f"{experiment_name}_predictions.csv"

    per_class_df.to_csv(per_class_path, index=False)
    cm_df.to_csv(cm_path)
    pred_df.to_csv(pred_path, index=False)

    if SAVE_CONFUSION_MATRIX_PNG:
        save_confusion_matrix_png(cm, experiment_name)

    print("\n" + "=" * 70)
    print(f"Final test results | {experiment_name}")
    print("=" * 70)

    for k, v in metrics.items():
        print(f"{k:24s}: {v:.6f}")

    print("\nPer-class metrics:")
    print(per_class_df)

    print("\nConfusion matrix:")
    print(cm_df)

    print("\nSaved:")
    print(" ", per_class_path)
    print(" ", cm_path)
    print(" ", pred_path)

    summary_row = {
        "experiment": experiment_name,
        "hardware": "A6000",
        "channel_mode": CHANNEL_MODE,
        "channels": ",".join(CHANNEL_NAMES),
        **metrics,
    }

    return summary_row


# ============================================================
# Training
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    y_train_original,
    epochs,
    lr,
    experiment_name,
):
    model = model.to(DEVICE)

    if USE_CLASS_WEIGHTS:
        class_weights = compute_class_weights(y_train_original).to(DEVICE)

        print("\nClass weights:")
        for act, w in zip(ACTIVITY_IDS, class_weights.detach().cpu().numpy()):
            print(f"  Activity {act}: {w:.4f}")

        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )

    # New non-deprecated GradScaler syntax.
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(USE_AMP and DEVICE == "cuda"),
    )

    best_val_macro_f1 = -1.0
    best_state = None
    epochs_without_improvement = 0

    history_rows = []

    for epoch in range(1, epochs + 1):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_seen = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                logits = model(X_batch)
                loss = criterion(logits, y_batch)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                pred = torch.argmax(logits, dim=1)
                correct = (pred == y_batch).sum().item()

            bs = len(y_batch)
            total_loss += loss.item() * bs
            total_correct += correct
            total_seen += bs

        train_loss = total_loss / max(total_seen, 1)
        train_accuracy = total_correct / max(total_seen, 1)

        val_true, val_pred, _ = evaluate_model(model, val_loader)
        val_metrics = compute_metrics(val_true, val_pred)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        }

        history_rows.append(row)

        print(
            f"{experiment_name} | "
            f"Epoch {epoch:03d}/{epochs} | "
            f"loss={train_loss:.4f} | "
            f"train_acc={train_accuracy:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
            f"val_bal_acc={val_metrics['balanced_accuracy']:.4f}",
            flush=True,
        )

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(
                f"Early stopping at epoch {epoch}. "
                f"Best val macro-F1 = {best_val_macro_f1:.4f}"
            )
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history_df = pd.DataFrame(history_rows)
    history_path = OUT_DIR / f"{experiment_name}_training_history.csv"
    history_df.to_csv(history_path, index=False)

    model_path = OUT_DIR / f"{experiment_name}_best_model.pt"

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "hardware": "A6000",
            "channel_mode": CHANNEL_MODE,
            "channels": CHANNEL_NAMES,
            "activity_ids": ACTIVITY_IDS,
            "window_len": WINDOW_LEN,
            "n_seq": N_SEQ,
            "n_steps": N_STEPS,
            "best_val_macro_f1": best_val_macro_f1,
            "batch_size": BATCH_SIZE,
            "use_amp": USE_AMP,
            "use_tf32": USE_TF32,
            "seed": SEED,
        },
        model_path,
    )

    print("Saved history:", history_path)
    print("Saved model:", model_path)

    return model, history_df


# ============================================================
# Experiment runner
# ============================================================

def train_once_and_evaluate(
    experiment_train_name,
    X_train,
    y_train,
    X_val,
    y_val,
    test_sets,
    epochs,
):
    print("\n" + "#" * 80)
    print(f"Training: {experiment_train_name}")
    print("#" * 80)
    print("Train X:", X_train.shape)
    print("Val X:  ", X_val.shape)
    print("Train label counts:", dict(zip(*np.unique(y_train, return_counts=True))))
    print("Val label counts:  ", dict(zip(*np.unique(y_val, return_counts=True))))

    train_loader = make_loader(
        X_train,
        y_train,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    val_loader = make_loader(
        X_val,
        y_val,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    model = make_new_model()

    model, history_df = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        y_train_original=y_train,
        epochs=epochs,
        lr=LR,
        experiment_name=experiment_train_name,
    )

    summary_rows = []

    for test_name, X_test, y_test in test_sets:
        print("\n" + "-" * 70)
        print(f"Testing trained model on: {test_name}")
        print("-" * 70)
        print("Test X:", X_test.shape)
        print("Test label counts:", dict(zip(*np.unique(y_test, return_counts=True))))

        test_loader = make_loader(
            X_test,
            y_test,
            batch_size=BATCH_SIZE,
            shuffle=False,
        )

        summary_row = save_final_results(
            model=model,
            test_loader=test_loader,
            experiment_name=test_name,
        )

        summary_rows.append(summary_row)

    return summary_rows


# ============================================================
# Main
# ============================================================

def main():
    print("=" * 80)
    print("CNN-BiLSTM v7 downstream evaluation | A6000 version")
    print("=" * 80)
    print("Device:", DEVICE)

    if DEVICE == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    print("CHANNEL_MODE:", CHANNEL_MODE)
    print("Using channels:", CHANNEL_NAMES)
    print("NUM_CHANNELS:", NUM_CHANNELS)
    print("BATCH_SIZE:", BATCH_SIZE)
    print("NUM_WORKERS:", NUM_WORKERS)
    print("USE_AMP:", USE_AMP)
    print("USE_TF32:", USE_TF32)
    print("SEED:", SEED)
    print("Output directory:", OUT_DIR)

    # ------------------------------------------------------------
    # Load data
    # ------------------------------------------------------------

    real_X, real_y, real_subjects = load_real_data()
    syn_X, syn_y, syn_subjects = load_synthetic_data()

    # ------------------------------------------------------------
    # Split real subjects
    # ------------------------------------------------------------

    train_subjects, val_subjects, test_subjects = make_subject_split(real_subjects)

    X_real_train, y_real_train, sub_real_train = filter_by_subjects(
        real_X,
        real_y,
        real_subjects,
        train_subjects,
    )

    X_real_val, y_real_val, sub_real_val = filter_by_subjects(
        real_X,
        real_y,
        real_subjects,
        val_subjects,
    )

    X_real_test, y_real_test, sub_real_test = filter_by_subjects(
        real_X,
        real_y,
        real_subjects,
        test_subjects,
    )

    print("\n" + "=" * 70)
    print("Final real split sizes")
    print("=" * 70)
    print("Real train:", X_real_train.shape)
    print("Real val:  ", X_real_val.shape)
    print("Real test: ", X_real_test.shape)
    print("Synthetic: ", syn_X.shape)

    split_size_df = pd.DataFrame([
        {
            "split": "real_train",
            "num_windows": len(X_real_train),
            "subjects": ",".join(train_subjects),
        },
        {
            "split": "real_val",
            "num_windows": len(X_real_val),
            "subjects": ",".join(val_subjects),
        },
        {
            "split": "real_test",
            "num_windows": len(X_real_test),
            "subjects": ",".join(test_subjects),
        },
        {
            "split": "synthetic_all",
            "num_windows": len(syn_X),
            "subjects": ",".join(sorted(np.unique(syn_subjects).astype(str))),
        },
    ])

    split_size_df.to_csv(OUT_DIR / "data_split_summary.csv", index=False)

    summary_rows = []

    # ------------------------------------------------------------
    # 1 and 2:
    # Train on real train subjects.
    # Test on real test subjects and synthetic data.
    # ------------------------------------------------------------

    summary_rows.extend(
        train_once_and_evaluate(
            experiment_train_name="train_real",
            X_train=X_real_train,
            y_train=y_real_train,
            X_val=X_real_val,
            y_val=y_real_val,
            test_sets=[
                ("real_train_to_real_test", X_real_test, y_real_test),
                ("real_train_to_synthetic_test", syn_X, syn_y),
            ],
            epochs=EPOCHS_REAL_TO_REAL,
        )
    )

    # ------------------------------------------------------------
    # 3:
    # Train on synthetic.
    # Test on real test subjects.
    #
    # Real validation is used for early stopping only.
    # ------------------------------------------------------------

    summary_rows.extend(
        train_once_and_evaluate(
            experiment_train_name="train_synthetic",
            X_train=syn_X,
            y_train=syn_y,
            X_val=X_real_val,
            y_val=y_real_val,
            test_sets=[
                ("synthetic_train_to_real_test", X_real_test, y_real_test),
            ],
            epochs=EPOCHS_SYN_TO_REAL,
        )
    )

    # ------------------------------------------------------------
    # 4:
    # Train on real train subjects + synthetic.
    # Test on real test subjects.
    # ------------------------------------------------------------

    X_real_plus_syn = np.concatenate([X_real_train, syn_X], axis=0).astype(np.float32)
    y_real_plus_syn = np.concatenate([y_real_train, syn_y], axis=0).astype(np.int64)

    summary_rows.extend(
        train_once_and_evaluate(
            experiment_train_name="train_real_plus_synthetic",
            X_train=X_real_plus_syn,
            y_train=y_real_plus_syn,
            X_val=X_real_val,
            y_val=y_real_val,
            test_sets=[
                ("real_plus_synthetic_train_to_real_test", X_real_test, y_real_test),
            ],
            epochs=EPOCHS_REAL_PLUS_SYN_TO_REAL,
        )
    )

    # ------------------------------------------------------------
    # Save summary
    # ------------------------------------------------------------

    summary_df = pd.DataFrame(summary_rows)

    metric_cols = [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_precision",
        "weighted_recall",
        "weighted_f1",
        "balanced_accuracy",
    ]

    first_cols = [
        "experiment",
        "hardware",
        "channel_mode",
        "channels",
    ]

    summary_df = summary_df[first_cols + metric_cols]

    summary_path = OUT_DIR / "summary_metrics.csv"
    summary_df.to_csv(summary_path, index=False)

    print("\n" + "=" * 80)
    print("ALL CNN-BiLSTM EXPERIMENTS DONE")
    print("=" * 80)
    print(summary_df)
    print("\nSaved summary:", summary_path)
    print("All outputs saved in:", OUT_DIR)


if __name__ == "__main__":
    main()

CNN-BiLSTM v7 downstream evaluation | A6000 version
Device: cuda
GPU: NVIDIA RTX A6000
CHANNEL_MODE: 6ch
Using channels: ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
NUM_CHANNELS: 6
BATCH_SIZE: 384
NUM_WORKERS: 6
USE_AMP: True
USE_TF32: True
SEED: 42
Output directory: /home/iailab42/khans1/projects/results/cnnbilstm_kovae_downstream_6ch

Loaded real data
X: (46907, 512, 6)
y: (46907,)
subjects: (46907,)
CHANNEL_MODE: 6ch
Using channels: ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
Label counts: {np.int64(1): np.int64(4534), np.int64(2): np.int64(3205), np.int64(3): np.int64(2277), np.int64(4): np.int64(3439), np.int64(5): np.int64(6807), np.int64(6): np.int64(13518), np.int64(7): np.int64(4660), np.int64(8): np.int64(8467)}
Subject counts: {np.str_('S1'): np.int64(3445), np.str_('S10'): np.int64(3532), np.str_('S11'): np.int64(3487), np.str_('S12'): np.int64(2938), np.str_('S13'): np.int64(3356), np.str_('S14'): np.int64(3177), np.str_('S15'): np.int64(2919), np.str_('S2'): n

In [2]:
# ============================================================
# evaluate_cnnbilstm_v7_a6000_random_subject_split.py
#
# CNN-BiLSTM downstream HAR evaluation for your v7 synthetic data.
#
# Main target:
#   NVIDIA RTX A6000 48GB
#
# Channel switch:
#   CHANNEL_MODE = "6ch" -> ACC_x, ACC_y, ACC_z, BVP, EDA, TEMP
#   CHANNEL_MODE = "4ch" -> ACC_x, ACC_y, ACC_z, BVP
#
# Subject split:
#   Random subject-wise split with fixed SEED.
#   With 15 subjects:
#       Train = 10 subjects
#       Val   = 2 subjects
#       Test  = 3 subjects
#
# Input real:
#   processed_all_subjects_64hz/all_X.npy
#   processed_all_subjects_64hz/all_y.npy
#   processed_all_subjects_64hz/all_subject.npy
#
# Input synthetic:
#   all6_average_synthetic_subject_outputs/generated_subjects_all_X.npy
#   all6_average_synthetic_subject_outputs/generated_subjects_all_y.npy
#   all6_average_synthetic_subject_outputs/generated_subjects_all_subject.npy
#
# Experiments:
#   1. Real train -> Real test
#   2. Real train -> Synthetic test
#   3. Synthetic train -> Real test
#   4. Real + Synthetic train -> Real test
#
# Outputs:
#   cnnbilstm_v7_results_a6000_6ch/
#   or
#   cnnbilstm_v7_results_a6000_4ch/
# ============================================================

import copy
import random
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support,
)


# ============================================================
# Main switch
# ============================================================

# Choose one:
CHANNEL_MODE = "6ch"
# CHANNEL_MODE = "4ch"


# ============================================================
# Paths
# ============================================================

PROJECT_ROOT = Path("/home/iailab42/khans1/projects")

REAL_DIR = PROJECT_ROOT / "data"
SYN_DIR = PROJECT_ROOT / "data" / "synthetic"

REAL_X_PATH = REAL_DIR / "all_X.npy"
REAL_Y_PATH = REAL_DIR / "all_y.npy"
REAL_SUBJECT_PATH = REAL_DIR / "all_subject.npy"

# SYN_X_PATH = SYN_DIR / "all_X_synthetic.npy"
# SYN_Y_PATH = SYN_DIR / "all_y_synthetic.npy"
# SYN_SUBJECT_PATH = SYN_DIR / "all_subject_synthetic.npy"

SYN_X_PATH = SYN_DIR / "timevae"/ "all_X_synthetic.npy"
SYN_Y_PATH = SYN_DIR / "timevae"/ "all_y_synthetic.npy"
SYN_SUBJECT_PATH = SYN_DIR / "timevae"/ "all_subject_synthetic.npy"
SYN_METADATA_PATH = SYN_DIR / "timevae"/ "all_metadata_synthetic.csv"

# OUT_DIR = PROJECT_ROOT / "results" / f"cnnbilstm_kovae_downstream_{CHANNEL_MODE}"
OUT_DIR = PROJECT_ROOT / "results" / f"cnnbilstm_timevae_downstream_{CHANNEL_MODE}"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# General config
# ============================================================

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ALL_CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]

if CHANNEL_MODE == "6ch":
    CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]
elif CHANNEL_MODE == "4ch":
    CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP"]
else:
    raise ValueError("CHANNEL_MODE must be '6ch' or '4ch'.")

USE_CHANNEL_INDICES = [ALL_CHANNEL_NAMES.index(ch) for ch in CHANNEL_NAMES]
NUM_CHANNELS = len(CHANNEL_NAMES)

ACTIVITY_IDS = [1, 2, 3, 4, 5, 6, 7, 8]
NUM_CLASSES = len(ACTIVITY_IDS)

WINDOW_LEN = 512

# Paper-style subsequence setup adapted to your 512-sample windows:
# paper: 128 = 4 x 32
# yours: 512 = 4 x 128
N_SEQ = 4
N_STEPS = WINDOW_LEN // N_SEQ

if WINDOW_LEN % N_SEQ != 0:
    raise ValueError("WINDOW_LEN must be divisible by N_SEQ.")

# Random subject-wise split counts.
N_TEST_SUBJECTS = 3
N_VAL_SUBJECTS = 2

# Leave empty for random split.
# Or manually set, e.g. TEST_SUBJECTS = ["S13", "S14", "S15"]
TEST_SUBJECTS = []
VAL_SUBJECTS = []


# ============================================================
# A6000 training settings
# ============================================================

BATCH_SIZE = 384
NUM_WORKERS = 6

EPOCHS_REAL_TO_REAL = 100
EPOCHS_REAL_TO_SYN = 100
EPOCHS_SYN_TO_REAL = 100
EPOCHS_REAL_PLUS_SYN_TO_REAL = 100

LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.5

USE_CLASS_WEIGHTS = True
USE_AMP = True
USE_TF32 = True

SAVE_CONFUSION_MATRIX_PNG = True

# Early stopping based on validation Macro-F1.
PATIENCE = 15


# ============================================================
# CUDA backend settings
# ============================================================

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = USE_TF32
    torch.backends.cudnn.allow_tf32 = USE_TF32

    # More reproducible. You can set True for speed, but False is safer.
    torch.backends.cudnn.benchmark = False


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


def autocast_context():
    """
    New non-deprecated AMP autocast syntax.

    Old deprecated:
        torch.cuda.amp.autocast()

    New:
        torch.amp.autocast("cuda", enabled=True/False)
    """
    if DEVICE == "cuda":
        return torch.amp.autocast("cuda", enabled=USE_AMP)

    return nullcontext()


# ============================================================
# Label helpers
# ============================================================

LABEL_TO_INDEX = {label: i for i, label in enumerate(ACTIVITY_IDS)}
INDEX_TO_LABEL = {i: label for label, i in LABEL_TO_INDEX.items()}


def labels_to_indices(y):
    y = np.asarray(y).astype(np.int64)
    out = np.zeros_like(y, dtype=np.int64)

    for i, lab in enumerate(y):
        lab = int(lab)

        if lab not in LABEL_TO_INDEX:
            raise ValueError(f"Unexpected label {lab}. Expected labels: {ACTIVITY_IDS}")

        out[i] = LABEL_TO_INDEX[lab]

    return out


def indices_to_labels(y_idx):
    y_idx = np.asarray(y_idx).astype(np.int64)
    return np.array([INDEX_TO_LABEL[int(i)] for i in y_idx], dtype=np.int64)


# ============================================================
# File helpers
# ============================================================

def require_file(path):
    path = Path(path)

    if not path.exists():
        error_message = (
            f"\nMissing file: {path}\n"
            f"Make sure these files exist:\n"
            f"  /home/iailab42/khans1/projects/data/all_X.npy\n"
            f"  /home/iailab42/khans1/projects/data/all_y.npy\n"
            f"  /home/iailab42/khans1/projects/data/all_subject.npy\n"
            f"  /home/iailab42/khans1/projects/data/synthetic/all_X_synthetic.npy\n"
            f"  /home/iailab42/khans1/projects/data/synthetic/all_y_synthetic.npy\n"
            f"  /home/iailab42/khans1/projects/data/synthetic/all_subject_synthetic.npy\n"
        )
        raise FileNotFoundError(error_message)

    return path


# ============================================================
# Load data
# ============================================================

def load_data(x_path, y_path, subject_path, name):
    require_file(x_path)
    require_file(y_path)
    require_file(subject_path)

    X = np.load(x_path).astype(np.float32)
    y = np.load(y_path).astype(np.int64)
    subjects = np.load(subject_path, allow_pickle=True).astype(str)

    if X.ndim != 3:
        raise ValueError(f"{name} X should be [N, T, C], got {X.shape}")

    if X.shape[1] != WINDOW_LEN:
        raise ValueError(
            f"{name} expected window length {WINDOW_LEN}, got {X.shape[1]}"
        )

    if X.shape[2] != len(ALL_CHANNEL_NAMES):
        raise ValueError(
            f"{name} expected original 6 channels, got {X.shape[2]}"
        )

    if len(X) != len(y):
        raise ValueError(f"{name} X/y mismatch: {len(X)} vs {len(y)}")

    if len(y) != len(subjects):
        raise ValueError(f"{name} y/subjects mismatch: {len(y)} vs {len(subjects)}")

    keep = np.isin(y, np.array(ACTIVITY_IDS, dtype=np.int64))

    X = X[keep]
    y = y[keep]
    subjects = subjects[keep]

    # Channel switch happens here.
    X = X[:, :, USE_CHANNEL_INDICES].astype(np.float32)

    print("\n" + "=" * 70)
    print(f"Loaded {name}")
    print("=" * 70)
    print("X:", X.shape)
    print("y:", y.shape)
    print("subjects:", subjects.shape)
    print("CHANNEL_MODE:", CHANNEL_MODE)
    print("Using channels:", CHANNEL_NAMES)
    print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))
    print("Subject counts:", dict(zip(*np.unique(subjects, return_counts=True))))

    return X, y, subjects


def load_real_data():
    return load_data(
        REAL_X_PATH,
        REAL_Y_PATH,
        REAL_SUBJECT_PATH,
        name="real data",
    )


def load_synthetic_data():
    return load_data(
        SYN_X_PATH,
        SYN_Y_PATH,
        SYN_SUBJECT_PATH,
        name="synthetic data",
    )


# ============================================================
# Subject split
# ============================================================

def subject_sort_key(s):
    s = str(s)

    if s.startswith("S"):
        try:
            return int(s[1:])
        except Exception:
            return s

    return s


def make_subject_split(real_subjects):
    unique_subjects = sorted(np.unique(real_subjects).astype(str), key=subject_sort_key)

    if len(unique_subjects) < (N_TEST_SUBJECTS + N_VAL_SUBJECTS + 1):
        raise ValueError(
            f"Not enough real subjects for split. Found: {unique_subjects}"
        )

    # Manual split if user fills TEST_SUBJECTS / VAL_SUBJECTS.
    if len(TEST_SUBJECTS) > 0 or len(VAL_SUBJECTS) > 0:
        if len(TEST_SUBJECTS) == 0 or len(VAL_SUBJECTS) == 0:
            raise ValueError(
                "If using manual split, set both TEST_SUBJECTS and VAL_SUBJECTS."
            )

        test_subjects = list(TEST_SUBJECTS)
        val_subjects = list(VAL_SUBJECTS)

        train_subjects = [
            s for s in unique_subjects
            if s not in test_subjects and s not in val_subjects
        ]

        split_mode = "manual"

    else:
        # Fixed random subject split.
        rng = np.random.default_rng(SEED)
        shuffled_subjects = np.array(unique_subjects, dtype=object)
        rng.shuffle(shuffled_subjects)
        shuffled_subjects = shuffled_subjects.tolist()

        test_subjects = shuffled_subjects[:N_TEST_SUBJECTS]
        val_subjects = shuffled_subjects[
            N_TEST_SUBJECTS:N_TEST_SUBJECTS + N_VAL_SUBJECTS
        ]
        train_subjects = shuffled_subjects[N_TEST_SUBJECTS + N_VAL_SUBJECTS:]

        split_mode = f"random_fixed_seed_{SEED}"

    train_subjects = sorted(train_subjects, key=subject_sort_key)
    val_subjects = sorted(val_subjects, key=subject_sort_key)
    test_subjects = sorted(test_subjects, key=subject_sort_key)

    if len(set(train_subjects) & set(val_subjects)) > 0:
        raise ValueError("Train/val subject overlap detected.")

    if len(set(train_subjects) & set(test_subjects)) > 0:
        raise ValueError("Train/test subject overlap detected.")

    if len(set(val_subjects) & set(test_subjects)) > 0:
        raise ValueError("Val/test subject overlap detected.")

    print("\n" + "=" * 70)
    print("Real subject split")
    print("=" * 70)
    print("Split mode:     ", split_mode)
    print("Train subjects: ", train_subjects)
    print("Val subjects:   ", val_subjects)
    print("Test subjects:  ", test_subjects)
    print("Train % by subject:", len(train_subjects) / len(unique_subjects) * 100.0)
    print("Val % by subject:  ", len(val_subjects) / len(unique_subjects) * 100.0)
    print("Test % by subject: ", len(test_subjects) / len(unique_subjects) * 100.0)

    split_df = pd.DataFrame({
        "split": ["train", "val", "test"],
        "subjects": [
            ",".join(train_subjects),
            ",".join(val_subjects),
            ",".join(test_subjects),
        ],
        "num_subjects": [
            len(train_subjects),
            len(val_subjects),
            len(test_subjects),
        ],
        "percentage_by_subject": [
            len(train_subjects) / len(unique_subjects) * 100.0,
            len(val_subjects) / len(unique_subjects) * 100.0,
            len(test_subjects) / len(unique_subjects) * 100.0,
        ],
        "split_mode": [split_mode, split_mode, split_mode],
        "seed": [SEED, SEED, SEED],
    })

    split_df.to_csv(OUT_DIR / "real_subject_split.csv", index=False)

    return train_subjects, val_subjects, test_subjects


def filter_by_subjects(X, y, subjects, selected_subjects):
    selected_subjects = set([str(s) for s in selected_subjects])
    mask = np.array([str(s) in selected_subjects for s in subjects], dtype=bool)

    return X[mask], y[mask], subjects[mask]


# ============================================================
# Dataset / DataLoader
# ============================================================

class HARWindowDataset(Dataset):
    def __init__(self, X, y_original):
        X = np.asarray(X, dtype=np.float32)
        y_original = np.asarray(y_original, dtype=np.int64)
        y_idx = labels_to_indices(y_original)

        if X.ndim != 3:
            raise ValueError(f"Expected X [N,T,C], got {X.shape}")

        if X.shape[1] != WINDOW_LEN:
            raise ValueError(f"Expected window length {WINDOW_LEN}, got {X.shape[1]}")

        if X.shape[2] != NUM_CHANNELS:
            raise ValueError(f"Expected {NUM_CHANNELS} channels, got {X.shape[2]}")

        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y_idx).long()
        self.y_original = y_original

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loader(X, y, batch_size, shuffle):
    dataset = HARWindowDataset(X, y)

    # Avoid BatchNorm crash from final train batch of size 1.
    drop_last = bool(shuffle and len(dataset) > batch_size)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=drop_last,
        persistent_workers=(NUM_WORKERS > 0),
    )

    return loader


def compute_class_weights(y_original):
    y_idx = labels_to_indices(y_original)
    counts = np.bincount(y_idx, minlength=NUM_CLASSES).astype(np.float32)

    counts = np.maximum(counts, 1.0)

    weights = counts.sum() / (NUM_CLASSES * counts)
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32)


# ============================================================
# CNN-BiLSTM model
# ============================================================

class TimeDistributedConvBranch(nn.Module):
    def __init__(self, in_channels, kernel_size, dropout):
        super().__init__()

        padding = kernel_size // 2

        self.net = nn.Sequential(
            nn.Conv1d(
                in_channels=in_channels,
                out_channels=64,
                kernel_size=kernel_size,
                padding=padding,
            ),
            nn.ReLU(inplace=True),

            nn.Conv1d(
                in_channels=64,
                out_channels=32,
                kernel_size=kernel_size,
                padding=padding,
            ),
            nn.ReLU(inplace=True),

            nn.Dropout(dropout),
            nn.MaxPool1d(kernel_size=2),
        )

    def forward(self, x):
        # x: [B, S, steps, C]
        B, S, steps, C = x.shape

        # TimeDistributed Conv1D:
        # [B, S, steps, C] -> [B*S, C, steps]
        x = x.reshape(B * S, steps, C)
        x = x.transpose(1, 2)

        h = self.net(x)

        # h: [B*S, 32, steps/2]
        h = h.reshape(B, S, -1)

        return h


class MultiBranchCNNBiLSTM(nn.Module):
    def __init__(
        self,
        num_channels,
        num_classes,
        n_seq,
        n_steps,
        dropout,
    ):
        super().__init__()

        self.num_channels = num_channels
        self.num_classes = num_classes
        self.n_seq = n_seq
        self.n_steps = n_steps

        self.branch_k3 = TimeDistributedConvBranch(
            in_channels=num_channels,
            kernel_size=3,
            dropout=dropout,
        )

        self.branch_k7 = TimeDistributedConvBranch(
            in_channels=num_channels,
            kernel_size=7,
            dropout=dropout,
        )

        self.branch_k11 = TimeDistributedConvBranch(
            in_channels=num_channels,
            kernel_size=11,
            dropout=dropout,
        )

        branch_out_dim = 32 * (n_steps // 2)
        lstm_input_dim = branch_out_dim * 3

        self.bilstm1 = nn.LSTM(
            input_size=lstm_input_dim,
            hidden_size=64,
            batch_first=True,
            bidirectional=True,
        )

        self.bilstm2 = nn.LSTM(
            input_size=64 * 2,
            hidden_size=32,
            batch_first=True,
            bidirectional=True,
        )

        self.fc = nn.Linear(32 * 2, 128)
        self.bn = nn.BatchNorm1d(128)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: [B, 512, C]
        B, T, C = x.shape

        expected_t = self.n_seq * self.n_steps

        if T != expected_t:
            raise ValueError(f"Expected T={expected_t}, got T={T}")

        if C != self.num_channels:
            raise ValueError(f"Expected C={self.num_channels}, got C={C}")

        # [B, 512, C] -> [B, 4, 128, C]
        x = x.reshape(B, self.n_seq, self.n_steps, C)

        h3 = self.branch_k3(x)
        h7 = self.branch_k7(x)
        h11 = self.branch_k11(x)

        h = torch.cat([h3, h7, h11], dim=-1)

        h, _ = self.bilstm1(h)
        h, (h_n, _) = self.bilstm2(h)

        # h_n shape: [num_layers * num_directions, B, hidden]
        # last forward = h_n[-2], last backward = h_n[-1]
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)

        z = self.fc(h_final)
        z = self.bn(z)
        z = F.relu(z)
        z = self.dropout(z)

        logits = self.classifier(z)

        return logits


def make_new_model():
    model = MultiBranchCNNBiLSTM(
        num_channels=NUM_CHANNELS,
        num_classes=NUM_CLASSES,
        n_seq=N_SEQ,
        n_steps=N_STEPS,
        dropout=DROPOUT,
    )

    return model


# ============================================================
# Metrics
# ============================================================

def evaluate_model(model, loader):
    model.eval()

    all_true = []
    all_pred = []
    all_prob = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            with autocast_context():
                logits = model(X_batch)

            probs = torch.softmax(logits.float(), dim=1)
            pred = torch.argmax(probs, dim=1)

            all_true.append(y_batch.cpu().numpy())
            all_pred.append(pred.cpu().numpy())
            all_prob.append(probs.cpu().numpy())

    y_true_idx = np.concatenate(all_true)
    y_pred_idx = np.concatenate(all_pred)
    y_prob = np.concatenate(all_prob)

    y_true = indices_to_labels(y_true_idx)
    y_pred = indices_to_labels(y_pred_idx)

    return y_true, y_pred, y_prob


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    accuracy = accuracy_score(y_true, y_pred)

    macro_precision = precision_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="macro",
        zero_division=0,
    )

    macro_recall = recall_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="macro",
        zero_division=0,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="macro",
        zero_division=0,
    )

    weighted_precision = precision_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="weighted",
        zero_division=0,
    )

    weighted_recall = recall_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="weighted",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        average="weighted",
        zero_division=0,
    )

    # Balanced accuracy for multiclass = average recall across classes.
    # This is intentionally same definition as macro recall.
    balanced_accuracy = macro_recall

    metrics = {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
        "balanced_accuracy": balanced_accuracy,
    }

    return metrics


def compute_per_class_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=ACTIVITY_IDS,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred, labels=ACTIVITY_IDS)

    row_sum = cm.sum(axis=1)
    diag = np.diag(cm)

    per_class_accuracy = np.divide(
        diag,
        row_sum,
        out=np.zeros_like(diag, dtype=np.float64),
        where=row_sum != 0,
    )

    rows = []

    for i, act in enumerate(ACTIVITY_IDS):
        rows.append({
            "activity": act,
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "per_class_accuracy_same_as_recall": float(per_class_accuracy[i]),
            "support": int(support[i]),
        })

    per_class_df = pd.DataFrame(rows)

    return per_class_df, cm


# ============================================================
# Plot / save results
# ============================================================

def save_confusion_matrix_png(cm, experiment_name):
    fig, ax = plt.subplots(figsize=(8, 7))

    im = ax.imshow(cm, interpolation="nearest")
    fig.colorbar(im, ax=ax)

    ax.set_title(f"Confusion Matrix | {experiment_name}")
    ax.set_xlabel("Predicted activity")
    ax.set_ylabel("True activity")

    tick_marks = np.arange(len(ACTIVITY_IDS))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels([str(a) for a in ACTIVITY_IDS])
    ax.set_yticklabels([str(a) for a in ACTIVITY_IDS])

    threshold = cm.max() / 2.0 if cm.max() > 0 else 0.5

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = int(cm[i, j])
            ax.text(
                j,
                i,
                str(value),
                ha="center",
                va="center",
                color="white" if value > threshold else "black",
                fontsize=9,
            )

    plt.tight_layout()

    save_path = OUT_DIR / f"{experiment_name}_confusion_matrix.png"
    plt.savefig(save_path, dpi=150)
    plt.close()

    print("Saved:", save_path)


def save_final_results(model, test_loader, experiment_name):
    y_true, y_pred, y_prob = evaluate_model(model, test_loader)

    metrics = compute_metrics(y_true, y_pred)
    per_class_df, cm = compute_per_class_metrics(y_true, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{a}" for a in ACTIVITY_IDS],
        columns=[f"pred_{a}" for a in ACTIVITY_IDS],
    )

    pred_df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "correct": y_true == y_pred,
    })

    for i, act in enumerate(ACTIVITY_IDS):
        pred_df[f"prob_activity_{act}"] = y_prob[:, i]

    per_class_path = OUT_DIR / f"{experiment_name}_per_class_metrics.csv"
    cm_path = OUT_DIR / f"{experiment_name}_confusion_matrix.csv"
    pred_path = OUT_DIR / f"{experiment_name}_predictions.csv"

    per_class_df.to_csv(per_class_path, index=False)
    cm_df.to_csv(cm_path)
    pred_df.to_csv(pred_path, index=False)

    if SAVE_CONFUSION_MATRIX_PNG:
        save_confusion_matrix_png(cm, experiment_name)

    print("\n" + "=" * 70)
    print(f"Final test results | {experiment_name}")
    print("=" * 70)

    for k, v in metrics.items():
        print(f"{k:24s}: {v:.6f}")

    print("\nPer-class metrics:")
    print(per_class_df)

    print("\nConfusion matrix:")
    print(cm_df)

    print("\nSaved:")
    print(" ", per_class_path)
    print(" ", cm_path)
    print(" ", pred_path)

    summary_row = {
        "experiment": experiment_name,
        "hardware": "A6000",
        "channel_mode": CHANNEL_MODE,
        "channels": ",".join(CHANNEL_NAMES),
        **metrics,
    }

    return summary_row


# ============================================================
# Training
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    y_train_original,
    epochs,
    lr,
    experiment_name,
):
    model = model.to(DEVICE)

    if USE_CLASS_WEIGHTS:
        class_weights = compute_class_weights(y_train_original).to(DEVICE)

        print("\nClass weights:")
        for act, w in zip(ACTIVITY_IDS, class_weights.detach().cpu().numpy()):
            print(f"  Activity {act}: {w:.4f}")

        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )

    # New non-deprecated GradScaler syntax.
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(USE_AMP and DEVICE == "cuda"),
    )

    best_val_macro_f1 = -1.0
    best_state = None
    epochs_without_improvement = 0

    history_rows = []

    for epoch in range(1, epochs + 1):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_seen = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                logits = model(X_batch)
                loss = criterion(logits, y_batch)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                pred = torch.argmax(logits, dim=1)
                correct = (pred == y_batch).sum().item()

            bs = len(y_batch)
            total_loss += loss.item() * bs
            total_correct += correct
            total_seen += bs

        train_loss = total_loss / max(total_seen, 1)
        train_accuracy = total_correct / max(total_seen, 1)

        val_true, val_pred, _ = evaluate_model(model, val_loader)
        val_metrics = compute_metrics(val_true, val_pred)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        }

        history_rows.append(row)

        print(
            f"{experiment_name} | "
            f"Epoch {epoch:03d}/{epochs} | "
            f"loss={train_loss:.4f} | "
            f"train_acc={train_accuracy:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
            f"val_bal_acc={val_metrics['balanced_accuracy']:.4f}",
            flush=True,
        )

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(
                f"Early stopping at epoch {epoch}. "
                f"Best val macro-F1 = {best_val_macro_f1:.4f}"
            )
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history_df = pd.DataFrame(history_rows)
    history_path = OUT_DIR / f"{experiment_name}_training_history.csv"
    history_df.to_csv(history_path, index=False)

    model_path = OUT_DIR / f"{experiment_name}_best_model.pt"

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "hardware": "A6000",
            "channel_mode": CHANNEL_MODE,
            "channels": CHANNEL_NAMES,
            "activity_ids": ACTIVITY_IDS,
            "window_len": WINDOW_LEN,
            "n_seq": N_SEQ,
            "n_steps": N_STEPS,
            "best_val_macro_f1": best_val_macro_f1,
            "batch_size": BATCH_SIZE,
            "use_amp": USE_AMP,
            "use_tf32": USE_TF32,
            "seed": SEED,
        },
        model_path,
    )

    print("Saved history:", history_path)
    print("Saved model:", model_path)

    return model, history_df


# ============================================================
# Experiment runner
# ============================================================

def train_once_and_evaluate(
    experiment_train_name,
    X_train,
    y_train,
    X_val,
    y_val,
    test_sets,
    epochs,
):
    print("\n" + "#" * 80)
    print(f"Training: {experiment_train_name}")
    print("#" * 80)
    print("Train X:", X_train.shape)
    print("Val X:  ", X_val.shape)
    print("Train label counts:", dict(zip(*np.unique(y_train, return_counts=True))))
    print("Val label counts:  ", dict(zip(*np.unique(y_val, return_counts=True))))

    train_loader = make_loader(
        X_train,
        y_train,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    val_loader = make_loader(
        X_val,
        y_val,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    model = make_new_model()

    model, history_df = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        y_train_original=y_train,
        epochs=epochs,
        lr=LR,
        experiment_name=experiment_train_name,
    )

    summary_rows = []

    for test_name, X_test, y_test in test_sets:
        print("\n" + "-" * 70)
        print(f"Testing trained model on: {test_name}")
        print("-" * 70)
        print("Test X:", X_test.shape)
        print("Test label counts:", dict(zip(*np.unique(y_test, return_counts=True))))

        test_loader = make_loader(
            X_test,
            y_test,
            batch_size=BATCH_SIZE,
            shuffle=False,
        )

        summary_row = save_final_results(
            model=model,
            test_loader=test_loader,
            experiment_name=test_name,
        )

        summary_rows.append(summary_row)

    return summary_rows


# ============================================================
# Main
# ============================================================

def main():
    print("=" * 80)
    print("CNN-BiLSTM v7 downstream evaluation | A6000 version")
    print("=" * 80)
    print("Device:", DEVICE)

    if DEVICE == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    print("CHANNEL_MODE:", CHANNEL_MODE)
    print("Using channels:", CHANNEL_NAMES)
    print("NUM_CHANNELS:", NUM_CHANNELS)
    print("BATCH_SIZE:", BATCH_SIZE)
    print("NUM_WORKERS:", NUM_WORKERS)
    print("USE_AMP:", USE_AMP)
    print("USE_TF32:", USE_TF32)
    print("SEED:", SEED)
    print("Output directory:", OUT_DIR)

    # ------------------------------------------------------------
    # Load data
    # ------------------------------------------------------------

    real_X, real_y, real_subjects = load_real_data()
    syn_X, syn_y, syn_subjects = load_synthetic_data()

    # ------------------------------------------------------------
    # Split real subjects
    # ------------------------------------------------------------

    train_subjects, val_subjects, test_subjects = make_subject_split(real_subjects)

    X_real_train, y_real_train, sub_real_train = filter_by_subjects(
        real_X,
        real_y,
        real_subjects,
        train_subjects,
    )

    X_real_val, y_real_val, sub_real_val = filter_by_subjects(
        real_X,
        real_y,
        real_subjects,
        val_subjects,
    )

    X_real_test, y_real_test, sub_real_test = filter_by_subjects(
        real_X,
        real_y,
        real_subjects,
        test_subjects,
    )

    print("\n" + "=" * 70)
    print("Final real split sizes")
    print("=" * 70)
    print("Real train:", X_real_train.shape)
    print("Real val:  ", X_real_val.shape)
    print("Real test: ", X_real_test.shape)
    print("Synthetic: ", syn_X.shape)

    split_size_df = pd.DataFrame([
        {
            "split": "real_train",
            "num_windows": len(X_real_train),
            "subjects": ",".join(train_subjects),
        },
        {
            "split": "real_val",
            "num_windows": len(X_real_val),
            "subjects": ",".join(val_subjects),
        },
        {
            "split": "real_test",
            "num_windows": len(X_real_test),
            "subjects": ",".join(test_subjects),
        },
        {
            "split": "synthetic_all",
            "num_windows": len(syn_X),
            "subjects": ",".join(sorted(np.unique(syn_subjects).astype(str))),
        },
    ])

    split_size_df.to_csv(OUT_DIR / "data_split_summary.csv", index=False)

    summary_rows = []

    # ------------------------------------------------------------
    # 1 and 2:
    # Train on real train subjects.
    # Test on real test subjects and synthetic data.
    # ------------------------------------------------------------

    summary_rows.extend(
        train_once_and_evaluate(
            experiment_train_name="train_real",
            X_train=X_real_train,
            y_train=y_real_train,
            X_val=X_real_val,
            y_val=y_real_val,
            test_sets=[
                ("real_train_to_real_test", X_real_test, y_real_test),
                ("real_train_to_synthetic_test", syn_X, syn_y),
            ],
            epochs=EPOCHS_REAL_TO_REAL,
        )
    )

    # ------------------------------------------------------------
    # 3:
    # Train on synthetic.
    # Test on real test subjects.
    #
    # Real validation is used for early stopping only.
    # ------------------------------------------------------------

    summary_rows.extend(
        train_once_and_evaluate(
            experiment_train_name="train_synthetic",
            X_train=syn_X,
            y_train=syn_y,
            X_val=X_real_val,
            y_val=y_real_val,
            test_sets=[
                ("synthetic_train_to_real_test", X_real_test, y_real_test),
            ],
            epochs=EPOCHS_SYN_TO_REAL,
        )
    )

    # ------------------------------------------------------------
    # 4:
    # Train on real train subjects + synthetic.
    # Test on real test subjects.
    # ------------------------------------------------------------

    X_real_plus_syn = np.concatenate([X_real_train, syn_X], axis=0).astype(np.float32)
    y_real_plus_syn = np.concatenate([y_real_train, syn_y], axis=0).astype(np.int64)

    summary_rows.extend(
        train_once_and_evaluate(
            experiment_train_name="train_real_plus_synthetic",
            X_train=X_real_plus_syn,
            y_train=y_real_plus_syn,
            X_val=X_real_val,
            y_val=y_real_val,
            test_sets=[
                ("real_plus_synthetic_train_to_real_test", X_real_test, y_real_test),
            ],
            epochs=EPOCHS_REAL_PLUS_SYN_TO_REAL,
        )
    )

    # ------------------------------------------------------------
    # Save summary
    # ------------------------------------------------------------

    summary_df = pd.DataFrame(summary_rows)

    metric_cols = [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_precision",
        "weighted_recall",
        "weighted_f1",
        "balanced_accuracy",
    ]

    first_cols = [
        "experiment",
        "hardware",
        "channel_mode",
        "channels",
    ]

    summary_df = summary_df[first_cols + metric_cols]

    summary_path = OUT_DIR / "summary_metrics.csv"
    summary_df.to_csv(summary_path, index=False)

    print("\n" + "=" * 80)
    print("ALL CNN-BiLSTM EXPERIMENTS DONE")
    print("=" * 80)
    print(summary_df)
    print("\nSaved summary:", summary_path)
    print("All outputs saved in:", OUT_DIR)


if __name__ == "__main__":
    main()

CNN-BiLSTM v7 downstream evaluation | A6000 version
Device: cuda
GPU: NVIDIA RTX A6000
CHANNEL_MODE: 6ch
Using channels: ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
NUM_CHANNELS: 6
BATCH_SIZE: 384
NUM_WORKERS: 6
USE_AMP: True
USE_TF32: True
SEED: 42
Output directory: /home/iailab42/khans1/projects/results/cnnbilstm_timevae_downstream_6ch

Loaded real data
X: (46907, 512, 6)
y: (46907,)
subjects: (46907,)
CHANNEL_MODE: 6ch
Using channels: ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
Label counts: {np.int64(1): np.int64(4534), np.int64(2): np.int64(3205), np.int64(3): np.int64(2277), np.int64(4): np.int64(3439), np.int64(5): np.int64(6807), np.int64(6): np.int64(13518), np.int64(7): np.int64(4660), np.int64(8): np.int64(8467)}
Subject counts: {np.str_('S1'): np.int64(3445), np.str_('S10'): np.int64(3532), np.str_('S11'): np.int64(3487), np.str_('S12'): np.int64(2938), np.str_('S13'): np.int64(3356), np.str_('S14'): np.int64(3177), np.str_('S15'): np.int64(2919), np.str_('S2'):